In [6]:
import pandas as pd
import h5py
import numpy as np
from datetime import timedelta

# --- 1. KONFIGURASI PATH ---
CSV_STEAD = "/Volumes/Local Disk/Code_Git/S3_code/seismic/data/merge.csv"
H5_STEAD  = "/Volumes/Local Disk/Code_Git/S3_code/seismic/data/merge.hdf5"
PATH_USGS = '/Volumes/Extreme SSD/benchmark_bmkg_indonesia/usgs_katalog/katalog_usgs_master_2001_2025.csv'
PATH_BMKG = '/Volumes/Extreme SSD/Indonesian Earthquake Catalog (BMKG), 1998–2024/BMKG_Earthquake_Catalog.csv'

# --- 2. DEFINISI RC BOUNDARIES ---
rc_data = {
    'RC_Code': ['RC_01', 'RC_02', 'RC_03', 'RC_04', 'RC_05', 'RC_06', 'RC_07', 'RC_08', 'RC_09', 'RC_10'],
    'Lat_Min': [0.5, 3.0, 7.0, 0.0, 6.0, 1.0, 0.0, 7.0, 0.0, 0.5],
    'Lat_Max': [6.0, 14.0, 14.0, 7.5, 7.5, 3.5, 14.0, 14.0, 7.5, 6.0],
    'Lon_Min': [92.0, 92.0, 113.5, 113.5, 132.0, 92.0, 108.5, 122.0, 123.5, 108.5],
    'Lon_Max': [109.0, 109.0, 122.5, 124.0, 141.0, 109.0, 114.0, 141.0, 132.5, 132.5]
}
df_rc = pd.DataFrame(rc_data)

# --- 3. FUNGSI HELPER ---
def assign_rc(lat, lon, df_rc_ref):
    """Menentukan kode RC berdasarkan koordinat latitude & longitude."""
    matched = [row['RC_Code'] for _, row in df_rc_ref.iterrows() 
               if (row['Lat_Min'] <= lat <= row['Lat_Max']) and (row['Lon_Min'] <= lon <= row['Lon_Max'])]
    return ",".join(matched) if matched else "Outside"

def match_events(df_base, df_ref, label_ref):
    """Mencari kecocokan event berdasarkan waktu (toleransi 60 detik)."""
    results = []
    # Loop untuk 500 data pertama agar proses tidak terlalu lama (bisa disesuaikan)
    for _, s_row in df_base.head(500).iterrows():
        t_start = s_row['origin_time'] - timedelta(seconds=60)
        t_end   = s_row['origin_time'] + timedelta(seconds=60)
        
        matches = df_ref[(df_ref['datetime'] >= t_start) & (df_ref['datetime'] <= t_end)]
        
        for _, r_row in matches.iterrows():
            results.append({
                'trace_name': s_row['trace_name'],
                'rc': s_row['assigned_rc'],
                'mag_stead': s_row['source_magnitude'],
                f'mag_{label_ref}': r_row['mag'],
                'diff_mag': s_row['source_magnitude'] - r_row['mag']
            })
    return pd.DataFrame(results)


# --- 4. PROSES EKSEKUSI ---

# A. Load & Filter STEAD
print("Memuat data STEAD...")
df_stead = pd.read_csv(CSV_STEAD)

# Perbaikan: Menggunakan nama kolom yang benar sesuai hasil print-mu
# Kita ubah namanya secara global agar konsisten dengan fungsi matching kita
df_stead.rename(columns={'source_origin_time': 'origin_time'}, inplace=True)

# Konversi ke datetime
df_stead['origin_time'] = pd.to_datetime(df_stead['origin_time'])

# Filter menggunakan nama kolom yang tepat
df_nf = df_stead[
    (df_stead['trace_category'] == "earthquake_local") &
    (df_stead['source_distance_km'] <= 20) &
    (df_stead['source_magnitude'] >= 3.0)
].copy().reset_index(drop=True)

# Beri label RC pada STEAD menggunakan koordinat source
df_nf['assigned_rc'] = df_nf.apply(
    lambda x: assign_rc(x['source_latitude'], x['source_longitude'], df_rc), 
    axis=1
)

print(f"Filter selesai. Ditemukan {len(df_nf)} data yang cocok.")


# Beri label RC pada STEAD
df_nf['assigned_rc'] = df_nf.apply(lambda x: assign_rc(x['source_latitude'], x['source_longitude'], df_rc), axis=1)

# B. Load & Clean Katalog BMKG
print("Memuat data BMKG...")
df_bmkg = pd.read_csv(PATH_BMKG)

# 1. Menggabungkan Tanggal dan Waktu
# Kita gunakan nama kolom persis sesuai hasil print-mu
try:
    df_bmkg['datetime'] = pd.to_datetime(df_bmkg['Date'] + ' ' + df_bmkg['Time (UTC)'])
except Exception as e:
    print(f"Gagal menggabungkan waktu: {e}")
    # Backup: jika ada format aneh, kita coba konversi paksa
    df_bmkg['datetime'] = pd.to_datetime(df_bmkg['Date'].astype(str) + ' ' + df_bmkg['Time (UTC)'].astype(str), errors='coerce')

# 2. Standarisasi Nama Kolom
# Kita ubah agar sesuai dengan fungsi match_events (mag, latitude, longitude)
rename_map_bmkg = {
    'Magnitude': 'mag',
    'Latitude': 'latitude',
    'Longitude': 'longitude',
    'Depth (km)': 'depth'
}
df_bmkg.rename(columns=rename_map_bmkg, inplace=True)

# 3. Membersihkan data yang gagal konversi waktu (jika ada)
df_bmkg = df_bmkg.dropna(subset=['datetime'])

print(f"Katalog BMKG siap. Total: {len(df_bmkg)} event.")

# C. Load & Clean Katalog USGS
print("Memuat data USGS...")
df_usgs = pd.read_csv(PATH_USGS)

# Gunakan format ISO8601 dan paksa menjadi UTC agar konsisten
try:
    df_usgs['datetime'] = pd.to_datetime(df_usgs['time'], format='ISO8601', utc=True)
except Exception:
    # Jika versi Pandas kamu lama, gunakan cara ini:
    df_usgs['datetime'] = pd.to_datetime(df_usgs['time'], utc=True)

# Penting: Hilangkan info timezone agar bisa dibandingkan dengan katalog lain yang non-timezone
df_usgs['datetime'] = df_usgs['datetime'].dt.tz_localize(None)

print(f"Katalog USGS siap. Total: {len(df_usgs)} event.")

# D. Investigasi (Matching)
print("Sedang menginvestigasi kecocokan data...")
res_bmkg = match_events(df_nf, df_bmkg, "bmkg")
res_usgs = match_events(df_nf, df_usgs, "usgs")

# --- 5. OUTPUT ---
print("\n--- RINGKASAN INVESTIGASI ---")
print(f"Total STEAD Terfilter: {len(df_nf)}")
print(f"Kecocokan dengan BMKG: {len(res_bmkg)} event")
print(f"Kecocokan dengan USGS: {len(res_usgs)} event")

if not res_bmkg.empty:
    print("\nContoh Perbandingan Magnitudo (STEAD vs BMKG):")
    print(res_bmkg[['trace_name', 'rc', 'mag_stead', 'mag_bmkg', 'diff_mag']].head())

Memuat data STEAD...


/var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/ipykernel_10796/3927007093.py:54: DtypeWarning: Columns (7,11,13,14,24,25,26,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df_stead = pd.read_csv(CSV_STEAD)


Filter selesai. Ditemukan 2515 data yang cocok.
Memuat data BMKG...
Katalog BMKG siap. Total: 217807 event.
Memuat data USGS...
Katalog USGS siap. Total: 83045 event.
Sedang menginvestigasi kecocokan data...

--- RINGKASAN INVESTIGASI ---
Total STEAD Terfilter: 2515
Kecocokan dengan BMKG: 14 event
Kecocokan dengan USGS: 7 event

Contoh Perbandingan Magnitudo (STEAD vs BMKG):
                 trace_name       rc  mag_stead  mag_bmkg  diff_mag
0  AKH.GO_20140309113129_EV  Outside        3.3       4.4      -1.1
1  AKH.GO_20140309113132_EV  Outside        3.3       4.4      -1.1
2  AQU.MN_20090407041228_EV  Outside        3.0       3.3      -0.3
3  AQU.MN_20090410154612_EV  Outside        3.4       3.1       0.3
4  AQU.MN_20090413133558_EV  Outside        3.8       5.1      -1.3
